# Centralised — Supervised Fine-Tuning (SFT)

$$\mathcal{L}_{\mathrm{SFT}} = -\frac{1}{|y^{+}|}\sum_{t} \log \pi_{\theta}\left(y^{+}_{t} \mid q, c^{+}, y^{+}_{<t}\right)$$

Token-level cross-entropy on the preferred answer under the relevant snippet.
Prompt tokens are masked, so only answer tokens contribute.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes rouge-score
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_centralized_sft_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 151.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.4 MB/s eta 0:00:00
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_sft_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SEQ_LEN = 4096
MAX_SNIPPET_TOKENS = 2048
MAX_ANSWER_TOKENS = 512
MAX_NEW_TOKENS = 512

START_MODE = "adapter"
CONTINUE_FROM_EPOCH = 3
CONTINUE_EPOCHS = 1
GLOBAL_EPOCHS = 3
LOCAL_LR = 2e-4
LOCAL_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

random.seed(SEED)

def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)

def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()

def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch
from datasets import Dataset, concatenate_datasets
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForSeq2Seq


model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

collator = DataCollatorForSeq2Seq(model_tok, label_pad_token_id=-100, padding=True, return_tensors="pt")
KEEP_COLS = ["user_query", "correct_answer", "related_conversation_snippet"]


def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )

def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)


def build_user_content(row):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )


def make_tokenize_fn(system_prompt):
    def fn(example):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_content(example)},
        ]
        prompt_text = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = model_tok(prompt_text, add_special_tokens=False)["input_ids"]
        answer_ids = model_tok(
            str(example["correct_answer"]) + model_tok.eos_token,
            add_special_tokens=False, truncation=True, max_length=MAX_ANSWER_TOKENS,
        )["input_ids"]
        input_ids = (prompt_ids + answer_ids)[:MAX_SEQ_LEN]
        labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_SEQ_LEN]
        return {"input_ids": input_ids, "labels": labels, "attention_mask": [1] * len(input_ids)}
    return fn


client_data: Dict[Any, Dict[str, Any]] = {}
tokenized_parts = []
for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    system_prompt = build_system_prompt()
    ds = Dataset.from_pandas(p_train[KEEP_COLS].reset_index(drop=True))
    tokenized = ds.map(make_tokenize_fn(system_prompt), remove_columns=ds.column_names)
    client_data[pid] = {
        "tokenized": tokenized,
        "n": len(tokenized),
        "system_prompt": system_prompt,
    }
    tokenized_parts.append(tokenized)
    print(f"  persona {pid}: {len(tokenized)} train examples")

pooled_tokenized = concatenate_datasets(tokenized_parts)
print(f"Pooled train examples: {len(pooled_tokenized)}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

  persona 980: 31 train examples


Map:   0%|          | 0/21 [00:00<?, ? examples/s]

  persona 987: 21 train examples


Map:   0%|          | 0/23 [00:00<?, ? examples/s]

  persona 996: 23 train examples
Pooled train examples: 3870


In [5]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def load_continual_adapter(from_epoch):
    """Continual learning: load base model + a previously saved LoRA adapter (trainable)."""
    adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{from_epoch}"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter to continue from not found: {adapter_dir}")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    m.print_trainable_parameters()
    return m

def load_base_with_adapter():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = get_peft_model(base, lora_config)
    return m

def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}

def sft_train(model, tokenized, epochs, lr, batch_size=LOCAL_BATCH_SIZE):
    model.train()
    model.config.use_cache = False
    loader = DataLoader(tokenized, batch_size=batch_size, shuffle=True, collate_fn=collator)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)
    epoch_losses: List[float] = []
    n_batches = 0
    for _ in range(epochs):
        batch_losses: List[float] = []
        for batch in tqdm(loader, desc="Training Batch"):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            batch_losses.append(float(loss.detach().cpu()))
            n_batches += 1
        if batch_losses:
            epoch_losses.append(sum(batch_losses) / len(batch_losses))
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    mean_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else None
    return {"mean_loss": mean_loss, "epoch_losses": epoch_losses, "n_batches": n_batches}

@torch.no_grad()
def eval_mean_loss(model, tokenized, batch_size=LOCAL_BATCH_SIZE):
    model.eval()
    loader = DataLoader(tokenized, batch_size=batch_size, shuffle=False, collate_fn=collator)
    losses: List[float] = []
    for batch in loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            losses.append(float(model(**batch).loss.detach().cpu()))
    model.train()
    return sum(losses) / len(losses) if losses else None

In [6]:
if START_MODE == "adapter":
    model = load_continual_adapter(CONTINUE_FROM_EPOCH)
    _start_epoch = CONTINUE_FROM_EPOCH + 1
    _end_epoch = CONTINUE_FROM_EPOCH + CONTINUE_EPOCHS
    print(f"Continual learning from adapter_epoch_{CONTINUE_FROM_EPOCH}: "
          f"training epochs {_start_epoch}..{_end_epoch}")
else:
    model = load_base_with_adapter()
    _start_epoch = 1
    _end_epoch = GLOBAL_EPOCHS
    print(f"Fresh HuggingFace base model: training epochs {_start_epoch}..{_end_epoch}")

GLOBAL_DIR = RESULTS_DIR / "global"
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
epoch_adapter_dirs: List[str] = []
centralized_loss_log: Dict[str, Any] = {"method": "centralized_sft_global", "epochs": []}

for epoch in range(_start_epoch, _end_epoch + 1):
    epoch_loss = sft_train(model, pooled_tokenized, 1, LOCAL_LR)
    client_losses = []
    epoch_entry = {
        "epoch": epoch,
        "pooled_mean_loss": epoch_loss["mean_loss"],
        "pooled_epoch_losses": epoch_loss["epoch_losses"],
        "pooled_n_batches": epoch_loss["n_batches"],
    }
    centralized_loss_log["epochs"].append(epoch_entry)
    with open(GLOBAL_DIR / f"losses_epoch_{epoch}.json", "w", encoding="utf-8") as f:
        json.dump(epoch_entry, f, indent=2)

    epoch_dir = GLOBAL_DIR / f"adapter_epoch_{epoch}"
    model.save_pretrained(str(epoch_dir))
    model_tok.save_pretrained(str(epoch_dir))
    epoch_adapter_dirs.append(str(epoch_dir))
    loss_str = f"{epoch_entry['pooled_mean_loss']:.4f}" if epoch_entry["pooled_mean_loss"] is not None else "n/a"
    print(f"Epoch {epoch}/{GLOBAL_EPOCHS} done | pooled_loss={loss_str} | saved {epoch_dir.name}")

with open(GLOBAL_DIR / "training_losses.json", "w", encoding="utf-8") as f:
    json.dump(centralized_loss_log, f, indent=2)

global_state = clone_state(model)

global_adapter_dir = GLOBAL_DIR / "adapter"
model.save_pretrained(str(global_adapter_dir))
model_tok.save_pretrained(str(global_adapter_dir))

with open(GLOBAL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "centralized_sft_global",
        "subset": SUBSET_NAME,
        "client_personas": [int(p) for p in CLIENT_PERSONAS],
        "num_clients": len(CLIENT_PERSONAS),
        "min_val_rows": MIN_VAL_ROWS,
        "seed": SEED,
        "global_epochs": GLOBAL_EPOCHS,
        "start_mode": START_MODE,
        "continue_from_epoch": CONTINUE_FROM_EPOCH if START_MODE == "adapter" else None,
        "continue_epochs": CONTINUE_EPOCHS if START_MODE == "adapter" else None,
        "trained_epoch_range": [int(_start_epoch), int(_end_epoch)],
        "pooled_train_examples": len(pooled_tokenized),
        "local_lr": LOCAL_LR,
        "epoch_adapter_dirs": epoch_adapter_dirs,
        "final_adapter_dir": str(global_adapter_dir),
    }, f, indent=2)

print("Saved final global centralized SFT adapter to:", global_adapter_dir.resolve())

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
Continual learning from adapter_epoch_3: training epochs 4..4


Training Batch:   0%|          | 0/484 [00:00<?, ?it/s]

Epoch 4/3 done | pooled_loss=1.1999 | saved adapter_epoch_4
Saved final global centralized SFT adapter to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_sft_snippet/global/adapter


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def load_global_model(epoch=None):
    """Load the centralized global LoRA adapter. Use epoch=1..GLOBAL_EPOCHS for checkpoints."""
    if epoch is not None:
        adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{epoch}"
    else:
        adapter_dir = RESULTS_DIR / "global" / "adapter"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = True
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    return m, tok

print("Final global adapter:", RESULTS_DIR / "global" / "adapter")
print("Per-epoch checkpoints:", RESULTS_DIR / "global" / "adapter_epoch_<n>")
print("Per-persona val preds:", RESULTS_DIR / "persona_<id>" / "val_predictions.csv")